# 05: Fixed-path Neural ODE loss comparison

**Status, 22 August:** design and implementation written; adequacy and training unrun. Detailed fixed procedure: ../docs/neural_ode_operator_experiments.md §§1 to 3.

## Experiment

One target path is fitted repeatedly while model class, initialization, solver, observations, optimizer and update count remain fixed. Training loss is varied. This isolates which fitted path each discrepancy selects under restricted approximation.

Target $Y^\star:[0,1]\to\mathbb R^2$ is

$$
Y_1^\star(t)=\cos(2\pi t)+a(t)\cos(12\pi t),\qquad
Y_2^\star(t)=\sin(2\pi t)+a(t)\sin(12\pi t),
$$

where $a(t)=0.3\exp[-100(t-0.25)^2]$. Large loop gives global structure; oscillatory burst near $t=0.25$ gives local structure.

Fitted path is generated by

$$
h(0)=\eta,\qquad
\dot h(t)=f_\theta(\gamma(t),h(t)),\qquad
\widehat Y(t)=g_\phi(h(t)).
$$

$\eta$, vector field $f_\theta$ and affine decoder $g_\phi$ are learned. Fixed-step RK4 uses maximum step $1/512$.

Primary losses are sample MSE and elapsed-time $J_2$. Smooth-only $H^1$ is secondary. Global and local signature losses enter after primary fits pass adequacy checks.


## Factors fixed before results

| factor | values |
|---|---|
| target observations | uniform; $t_r=1-(1-r/63)^3$ clustered |
| capacity | restricted $(H=2,w=16)$; expressive $(H=8,w=64)$ |
| seed | $0,1,2,3,4$ |
| primary loss | MSE; $J_2$ |
| secondary loss | $H^1$, uniform observations only |
| updates | 5,000 full-batch Adam updates |

Every run is evaluated on 513 uniform points using dense MSE, $J_2$, $H^1$, supremum error and local $J_2$ on $[0.15,0.35]$. Equal seed and capacity must have identical saved initialization fingerprints across losses.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

RESULT_ROOT = Path('../results/runs/neural_ode_fixed_path')

def load_metrics(root=RESULT_ROOT):
    rows = []
    for path in root.glob('*/seed*/*/*/meta.json'):
        meta = json.loads(path.read_text())
        cfg = meta['fit_config']
        rows.append({
            'capacity': path.parts[-5],
            'seed': cfg['seed'],
            'condition': cfg['condition'],
            'loss': cfg['loss'],
            'fingerprint': meta['initial_fingerprint'],
            **meta['metrics'],
        })
    return pd.DataFrame(rows)

metrics = load_metrics()
metrics


## Acceptance before interpretation

1. Fourier adequacy file reports improvement over raw time and expressive dense MSE at most $10^{-3}$.
2. Initial fingerprints match across losses within every capacity and seed.
3. Every parameter receives finite gradient.
4. Training loss decreases and dense fitted paths are finite.
5. Uniform MSE and $J_2$ are expected to be close, with endpoint half-weight differences.

Results and interpretation are added only after these checks pass.
